# Set up imports

In [1]:
import logging
import numpy as np
from blastwave import BoomClient, BOOMCredentialsError

In [2]:
logger = logging.getLogger(__name__)

# Create Boom Client

In [3]:
boom = BoomClient()

# Try pinging server

In [4]:
try:
    boom.ping()
except BOOMCredentialsError as exc:
    logger.error("Error with querying BOOM using credentials")
    raise exc

# See list of available catalogs

In [5]:
boom.get_catalogs()

['2MASS_PSC',
 'CatWISE2020',
 'GALEX',
 'Gaia_DR3',
 'LSPSC',
 'LSST_alerts',
 'LSST_alerts_aux',
 'LSST_alerts_cutouts_20260519',
 'NED',
 'PS1_DR1',
 'TNS',
 'VSX',
 'ZTF_alerts',
 'ZTF_alerts_aux',
 'ZTF_alerts_cutouts_20260519',
 'milliquas_v8']

# Check a catalog

In [6]:
catalog = "milliquas_v8"
# catalog="ZTF_alerts_aux"

In [7]:
assert catalog in boom.get_catalogs(), f"Catalog {catalog} not listed in catalogs"

In [8]:
entry_count = boom.get_entry_count(catalog)
print(f"Catalog {catalog} has {entry_count} entries")

Catalog milliquas_v8 has 1021800 entries


In [9]:
indexes = boom.get_catalog_indexes(catalog)
print(f"Catalog {catalog} has the following index columns: {indexes}")

Catalog milliquas_v8 has the following index columns: [{'key': {'_id': 1}, 'name': '_id_', 'v': 2}, {'key': {'coordinates.radec_geojson': '2dsphere'}, 'name': 'coordinates.radec_geojson_2dsphere', 'v': 2, '2dsphereIndexVersion': 3}]


In [10]:
example = boom.get_sample_data(catalog)
print(f"Example entry from {catalog}:\n")
example

Example entry from milliquas_v8:



{'_id': 'SDSS J160354.46+085714.0',
 'ra': 240.9769147,
 'dec': 8.9538968,
 'objtype': 'QR',
 'rmag': 19.690000534057617,
 'bmag': 19.899999618530273,
 'comment': 'gG',
 'z': 1.350000023841858,
 'rname': 'VLAJ160354.46+085714.1',
 'lobe1': 'FIRST J160354.4+085714',
 'lobe2': 'RACS J160354.2+085715',
 'coordinates': {'radec_geojson': {'type': 'Point',
   'coordinates': [60.97691470000001, 8.9538968]}}}

# Query a catalog

BOOM uses MongoDB type queries. You can query on any of the fields in the sample data, but it'll be faster if you query on indexes 

In [11]:
# Here's an example, let's take the 3 sources closest to the above object. Look up MongoDB queries for syntax.
ra, dec = example["ra"], example["dec"]

mongodb_filter = {
    "coordinates.radec_geojson": {
        "$nearSphere": {
            "$geometry": {
                "type": "Point",
                "coordinates": [ra - 180.0, dec], # MongoDB convention
            },
        }
    }
}

query = {
    "catalog_name": catalog,
    "filter": mongodb_filter,
    "limit": 3, # Set limit of 3 sources
    "projection": { # Choose to only return some fields
        "source_name": 1,
        "ra": 1,
        "dec": 1
    }
}

In [12]:
boom.query(query)

[{'_id': 'SDSS J160354.46+085714.0', 'ra': 240.9769147, 'dec': 8.9538968},
 {'_id': 'SDSS J160414.39+085623.1', 'ra': 241.0599627, 'dec': 8.9397508},
 {'_id': 'SDSS J160338.76+090216.8', 'ra': 240.91152, 'dec': 9.0380101}]

We can also set a maximum radius, and count the number of objects in that radius:

In [13]:
radius_arcsec = 10.
    
radius_rad = np.radians(radius_arcsec / 3600.)

mongodb_filter = {
    "coordinates.radec_geojson": {
        "$nearSphere": {
            "$geometry": {
                "type": "Point",
                "coordinates": [ra - 180.0, dec], # MongoDB convention
            },
        },
        "$maxDistance": radius_rad#  * 6371008.8, # Conversion to sphere surface ('Earth m')
    }
}

query = {
    "catalog_name": catalog,
    "filter": mongodb_filter,
    "limit": 5, # Set limit of 3 sources
    "projection": { # Choose to only return some fields
        "source_name": 1,
        "ra": 1,
        "dec": 1
    }
}

In [14]:
# boom.count(query)
res = boom.api("post", "queries/count", data=query)
res.json()
# res.raise_for_status()

RetryError: HTTPSConnectionPool(host='api.kaboom.caltech.edu', port=443): Max retries exceeded with url: /queries/count (Caused by ResponseError('too many 500 error responses'))

In [ ]:
# We see the three closest objects, including our original object first in the list
res.json()["data"]

# See the full BOOM API documentation at https://api.kaboom.caltech.edu/docs 
# See MongoDB documentation at https://www.mongodb.com/docs/manual/reference/mql/query-predicates